<p align="center">
  <img src="fig/wrfhydro_logo.png" alt="WRF-Hydro Logo" width="15%">
  &nbsp;&nbsp;&nbsp; <!-- adds horizontal space -->
  <img src="fig/Coupling_preparation.png" alt="Coupling Preparation" width="70%">
</p>

# WRF-Hydro/MODFLOW Domain Setup: Step 3 — MODFLOW flopy Process and Package Modification

This Python notebook defines and creates flopy packates to generate a MODFLOW6 simulation for a subset grid.

## Authors

**First Version:** Built upon Wesley Zell (wzell@usgs.gov) CONUS MODFLOW subsetting tool (ngwm1.0_viz_and_subset.ipynb)  

**Created by:** Farshid Felfelani  
Research Applications Laboratory, NSF National Center for Atmospheric Research  

- **Email:** felfelan@ucar.edu


In [1]:
import numpy as np
import flopy
import os, re
import shutil
import rasterio
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

## 1. Setting ENV variables of the Notebook

In [2]:
# tag = 'ngwm1.0'
dxdy = 250

version = "01"

# case = "rep", "drb"
case = "rep"

tag = case + "_" + str(dxdy) + "m" + "_v" + version

subset_in = os.path.join("./modflow_cutout", tag, "modflow_subset")
subset_ot = os.path.join("./modflow_cutout", tag, "Run_dir")

if not os.path.exists(subset_ot):
    os.makedirs(subset_ot)

sim_ws = "./"

modelname = 'modflow_subset'

In [3]:

file_in = os.path.join(subset_in0, "wrfhydro_cutout", tag, "shp", "HGT_M.tif")
file_out = os.path.join(subset_ot, "HGT_M.tif")

if not os.path.isfile(file_out):
    shutil.copyfile(file_in, file_out)

raster_f = rasterio.open(os.path.join(subset_ot, 'HGT_M.tif'), 'r')
raster_f.read().shape

(1, 975, 2375)

In [4]:
def purge(dir, pattern):
    for f in os.listdir(dir):
        if re.search(pattern, f):
            os.remove(os.path.join(dir, f))

In [5]:
def copyfunc(dirin, pattern, dirout):
    for f in os.listdir(dirin):
        if re.search(pattern, f):
            print("COPYING ... ", os.path.join(dirin, f))
            shutil.copyfile(os.path.join(dirin, f), os.path.join(dirout, f))

In [6]:
def movefunc(dirin, pattern, dirout):
    for f in os.listdir(dirin):
        if re.search(pattern, f):
            print("MOVING ... ", os.path.join(dirin, f))
            shutil.move(os.path.join(dirin, f), os.path.join(dirout, f))

In [7]:
purge(subset_ot, "modflow_subset")

In [8]:
copyfunc(subset_in, "modflow_subset.top", sim_ws)
copyfunc(subset_in, "modflow_subset.bottoms", sim_ws)
copyfunc(subset_in, "modflow_subset.hk", sim_ws)
copyfunc(subset_in, "modflow_subset.idomain", sim_ws)
copyfunc(subset_in, "modflow_subset.rch", sim_ws)
copyfunc(subset_in, "modflow_subset.startingheads", sim_ws)


COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.top
COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.bottoms
COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.hk
COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.idomain
COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.rch
COPYING ...  /glade/work/felfelan/wrf_hydro_nwm_public/soren_bmi/coupled_prepostproc_tutorial/modflow_cutout/rep_250m_v01/modflow_subset/modflow_subset.startingheads


In [9]:
dtime = pd.DataFrame({'Date':pd.date_range(start='1-1-2021 00:00:00', end='1-31-2021 23:00:00', freq='D')})
days_in_month = dtime['Date'].dt.days_in_month

# Daily stress period and daily time-steps; unit is hours
tdis_rc = [(24.0, 1.0, 1.0) for ii in range(len(dtime))]
print(tdis_rc[:12])

[(24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0), (24.0, 1.0, 1.0)]


## 2. Flopy simulation setup and package definition

### 2.1. Initialize the MODFLOW6 simulation

In [11]:
sim = flopy.mf6.MFSimulation(sim_name=modelname, sim_ws=sim_ws, exe_name='../make/mf6', version='mf6')
tdis = flopy.mf6.ModflowTdis(sim, pname='tdis', time_units='hours', nper=len(dtime), perioddata=tdis_rc)

gwf = flopy.mf6.ModflowGwf(sim, modelname=modelname)
gwf.name_file.newtonoptions = 'UNDER_RELAXATION'

ims = flopy.mf6.ModflowIms(sim, pname='ims', print_option='SUMMARY', complexity='COMPLEX')
sim.register_ims_package(ims, [modelname])

oc_pckg = os.path.join(sim_ws, '{}.{}'.format(modelname, 'oc'))
cbc_output = os.path.join(sim_ws, '{}.{}'.format(modelname, 'cbc'))
hds_output = os.path.join(sim_ws, '{}.{}'.format(modelname, 'hds'))

oc = flopy.mf6.ModflowGwfoc(gwf, pname='oc', filename=oc_pckg, \
                            budget_filerecord=cbc_output, \
                            head_filerecord=hds_output, \
                            saverecord=[('HEAD', 'LAST'), ('BUDGET', 'LAST')])

### 2.2. Add the spatial framework (i.e., the discretization package) and initial conditions

In [12]:
dis_file = os.path.join(sim_ws, '{}.{}'.format(modelname, 'dis'))
idomain_file = os.path.join(sim_ws, '{}.{}'.format(modelname, 'idomain'))
top_file = os.path.join(sim_ws, '{}.{}'.format(modelname, 'top'))
bottoms_file = os.path.join(sim_ws, '{}.{}'.format(modelname, 'bottoms'))
strt_file = os.path.join(sim_ws, '{}.{}'.format(modelname, 'startingheads'))

dis = flopy.mf6.ModflowGwfdis(gwf, pname='dis', filename=dis_file, length_units='METERS', \
                              nlay=1, nrow=raster_f.read().shape[1], ncol=raster_f.read().shape[2], \
                              delr=250, delc=250, \
                              idomain=idomain_file, top=top_file, botm=bottoms_file)

ic_pckg = os.path.join(sim_ws, '{}.{}'.format(modelname, 'ic'))
start_file = top_file

ic = flopy.mf6.ModflowGwfic(gwf, pname='ic', filename=ic_pckg, strt=strt_file)

### 2.3. Add the hydraulic properties and forcing (i.e., recharge)

In [13]:
npf_pak_kwargs = {'pname': 'npf',
                  'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'npf')),
                  'save_flows': True,
                  'icelltype': [1], 'save_specific_discharge': True, 'save_saturation': True,
                  'k': {'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'hk'))},
                  'k33': 0.1}

npf = flopy.mf6.ModflowGwfnpf(gwf, **npf_pak_kwargs)

In [14]:
rch_pak_kwargs = {'pname': 'rch',
                  'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'rcha')),
                  'save_flows': True,
                  'recharge': {0: {'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'rch'))}}}

rch = flopy.mf6.ModflowGwfrcha(gwf, **rch_pak_kwargs)

The drain, general head, and constant head boundaries are a little trickier to write because we must translate the data arrays to list format.  
But still not too bad ...

In [15]:
def unwrap_arrays(filename, lay=0):
    """Helper to unwrap ASCII array to a list of row/col/values."""

    a0 = np.genfromtxt(filename)

    # NaN indicates that the boundary condition does not apply;
    # therefore, reduce the array to non-nan entries
    a = a0[np.isfinite(a0)]
    # a = a0[:]
    
    if len(a.flatten()) == 0:
        
        return None
    
    else:

        lays = [lay] * len(a.flatten())
        rows, cols = np.where(np.isfinite(a0))
        # rows, cols = np.where(a0)
        
        return [lays, rows, cols, a.flatten()]

drn_stages = unwrap_arrays(filename=os.path.join(subset_in, '{}.{}'.format(modelname, 'drn_stage')))
drn_conductances = unwrap_arrays(filename=os.path.join(subset_in, '{}.{}'.format(modelname, 'drn_conductance')))


top_stages = unwrap_arrays(filename=top_file)

# Combine the stages and conductances into a single list of tuples
drn_data = [(k, i, j, s, c) for k, i, j, s, c in zip(drn_stages[0],
                                                     drn_stages[1],
                                                     drn_stages[2],
                                                     drn_stages[3],
                                                     drn_conductances[3])]


ghb_stages = unwrap_arrays(filename=os.path.join(subset_in, '{}.{}'.format(modelname, 'ghb_stage')))
ghb_conductances = unwrap_arrays(filename=os.path.join(subset_in, '{}.{}'.format(modelname, 'ghb_conductance')))

top_unwrap =  unwrap_arrays(filename=top_file)

# Combine the stages and conductances into a single list of tuples
ghb_data = [(k, i, j, s, c) for k, i, j, s, c in zip(ghb_stages[0],
                                                     ghb_stages[1],
                                                     ghb_stages[2],
                                                     ghb_stages[3],
                                                     ghb_conductances[3])]

chd_stages = unwrap_arrays(filename=os.path.join(subset_in, '{}.{}'.format(modelname, 'chd_stage')))
chd_data = chd_stages  # No need to combine stage and conductance for CHD

In [16]:
drn_pak_kwargs = {'pname': 'drn',
                  'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'drn')),
                  'save_flows': True,
                  'maxbound': len(drn_data),
                  'stress_period_data': {0: drn_data}}

drn = flopy.mf6.ModflowGwfdrn(gwf, **drn_pak_kwargs)

In [17]:
ghb_pak_kwargs = {'pname': 'ghb',
                  'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'ghb')),
                  'save_flows': True,
                  'maxbound': len(ghb_data),
                  'stress_period_data': {0: ghb_data}}

ghb = flopy.mf6.ModflowGwfghb(gwf, **ghb_pak_kwargs)

In [18]:
sto_pak_kwargs = {'pname': 'sto',
                  'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'sto')),
                  'save_flows': True,
                  'ss': 1.00000000E-05,
                  'sy': 0.15,
                  'iconvert': 1}

sto = flopy.mf6.ModflowGwfsto(gwf, **sto_pak_kwargs)

In [19]:
if chd_data is not None:
    chd_pak_kwargs = {'pname': 'chd',
                      'filename': os.path.join(sim_ws, '{}.{}'.format(modelname, 'chd')),
                      'save_flows': True,
                      'maxbound': len(chd_data),
                      'stress_period_data': {0: chd_data}}

    chd = flopy.mf6.ModflowGwfchd(gwf, **chd_pak_kwargs)
    # pass # Replace this with block to write coastal constant head
    # FARSHID: NEED TO MAKE SURE PASS IS WHAT WE NEED; THE ABOVE BLOCK GIVES ERROR FOR THE DRB
    # I HAVE COMMENTED THE BLOCK AND UNCOMMENTED PASS

### 2.4. Write and execute the MODFLOW simulation

In [20]:
sim.write_simulation()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims...
  writing model modflow_subset...
    writing model name file...
    writing package oc...
    writing package dis...
    writing package ic...
    writing package npf...
    writing package rch...
    writing package drn...
    writing package ghb...
    writing package sto...


In [21]:
# gwf = sim.get_model()
gwf.get_package_list()

['OC', 'DIS', 'IC', 'NPF', 'RCH', 'DRN', 'GHB', 'STO']

In [23]:
movefunc("./", "modflow_subset", subset_ot)

MOVING ...  ./modflow_subset.npf
MOVING ...  ./modflow_subset.dis
MOVING ...  ./modflow_subset.idomain
MOVING ...  ./modflow_subset.ghb
MOVING ...  ./modflow_subset.ims
MOVING ...  ./modflow_subset.startingheads
MOVING ...  ./modflow_subset.top
MOVING ...  ./modflow_subset.ic
MOVING ...  ./modflow_subset.hk
MOVING ...  ./modflow_subset.sto
MOVING ...  ./modflow_subset.rcha
MOVING ...  ./modflow_subset.bottoms
MOVING ...  ./modflow_subset.tdis
MOVING ...  ./modflow_subset.rch
MOVING ...  ./modflow_subset.nam
MOVING ...  ./modflow_subset.drn
MOVING ...  ./modflow_subset.oc


In [24]:
movefunc("./", "mfsim", subset_ot)

MOVING ...  ./mfsim.nam
